# 09 — MCP with Agent Framework

**What you'll learn:**
- What role MCP plays in agent systems
- How to connect a real MCP server with Agent Framework
- How to use free/open-source MCP servers in notebooks
- How to keep domain capabilities external to your agent prompt

---

## Why MCP + Agent Framework?

Agent Framework is great at orchestration, tool-calling, and multi-agent design.
MCP (Model Context Protocol) gives you a standard way to connect external
data sources and capabilities (docs, tickets, internal APIs, etc.).

In practice, a common pattern is:
1. Agent decides a tool is needed
2. Agent Framework routes to an MCP tool
3. MCP server returns grounded data
4. Agent uses that data in the final response

In [2]:
import os
import shutil

from azure.identity.aio import DefaultAzureCredential
from dotenv import load_dotenv

from agent_framework import MCPStdioTool
from agent_framework.azure import AzureOpenAIResponsesClient

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
DEPLOYMENT_NAME = os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"]

if shutil.which("uvx") is None:
    raise RuntimeError(
        "uvx is required for this example. Install uv first: https://docs.astral.sh/uv/"
    )

## Real MCP Server (Open Source, Free)

This example uses the official open-source MCP time server over stdio:
`mcp-server-time`, launched via `uvx`.

Note: for this package, stdio is the default transport, so no extra
`--stdio` flag is required.

Agent Framework connects to it via `MCPStdioTool`, discovers available tools,
and exposes them to your agent automatically.

In [5]:
existing_tool = globals().get("mcp_time_tool")
if existing_tool is not None:
    try:
        await existing_tool.close()
    except Exception:
        pass

mcp_tool_call_count = 0


def parse_mcp_tool_result(result):
    global mcp_tool_call_count
    mcp_tool_call_count += 1
    print(f"[MCP] Tool result received from server (call #{mcp_tool_call_count})")

    text_parts = []
    for item in getattr(result, "content", []) or []:
        text_value = getattr(item, "text", None)
        if text_value:
            text_parts.append(text_value)

    return "\n".join(text_parts) if text_parts else str(result)


mcp_time_tool = MCPStdioTool(
    name="time_server",
    command="uvx",
    args=["mcp-server-time"],
    description="Time and timezone tools from an open-source MCP server",
    tool_name_prefix="time",
    approval_mode="never_require",
    load_prompts=False,
    parse_tool_results=parse_mcp_tool_result,
)

print("Configured MCP server tool: time_server")

Configured MCP server tool: time_server


## Create an Agent That Uses the Real MCP Tool

In [59]:
credential = DefaultAzureCredential()

agent = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=DEPLOYMENT_NAME,
    credential=credential,
).as_agent(
    name="MCPTimeAssistant",
    instructions=(
        "You are a precise assistant for date/time questions. "
        "Use the available MCP time tools for timezone conversions and current time. "
        "If a tool call fails, explain clearly and provide best-effort fallback guidance."
    ),
    tools=[mcp_time_tool],
)

print(f"Agent created: {agent.name}")

Agent created: MCPTimeAssistant


## Demo: Ask for Current Time in Another Region

In [60]:
before_calls = mcp_tool_call_count
result = await agent.run(
    "What time is it right now in Tokyo? Include timezone abbreviation and UTC offset."
)
after_calls = mcp_tool_call_count

print(result)
print(f"MCP tool calls during this run: {after_calls - before_calls}")

[MCP] Tool result received from server (call #1)
The current time in Tokyo is **11:50 AM** on **Wednesday, March 25, 2026**. The timezone is **JST (Japan Standard Time)** with a UTC offset of **+9 hours**.
MCP tool calls during this run: 1


## Demo: Timezone Conversion

In [61]:
before_calls = mcp_tool_call_count
result = await agent.run(
    "Convert 2026-03-24 09:00 America/New_York to Europe/London time and explain DST impact if relevant."
)
after_calls = mcp_tool_call_count

print(result)
print(f"MCP tool calls during this run: {after_calls - before_calls}")

[MCP] Tool result received from server (call #2)
At 09:00 AM on **March 24, 2026**, in **America/New_York**, the time in **Europe/London** will be **1:00 PM**.

### Explanation of Daylight Saving Time (DST) Impact:
1. In **America/New_York**, DST is active on March 24, 2026, so the local time is shifted forward by 1 hour. As a result, the UTC offset for New York is **-4:00** instead of the standard **-5:00**.
   
2. In **Europe/London**, DST is **not yet active** on this date. Their standard time (UTC+0:00) applies. DST typically begins in Europe on the last Sunday of March, which would be March 29, 2026.

### Time Difference:
Because of the overlapping effects, there is a time difference of **+4 hours** between America/New_York and Europe/London at this time.
MCP tool calls during this run: 1


## Verify MCP Tools Were Actually Called

This notebook logs MCP server usage from `parse_mcp_tool_result`.

What to look for:
- Lines like `[MCP] Tool result received from server (call #N)`
- `MCP tool calls during this run: X` after each demo prompt

If `X > 0`, the agent invoked tools through MCP for that run.

In [62]:
print(f"Total MCP tool calls observed in this kernel session: {mcp_tool_call_count}")

# Optional: inspect discovered MCP tool names
if not mcp_time_tool.is_connected:
    await mcp_time_tool.connect()

print("Discovered MCP tools:")
for i, fn in enumerate(mcp_time_tool.functions, 1):
    print(f"{i}. {fn.name}")

Total MCP tool calls observed in this kernel session: 2
Discovered MCP tools:
1. time_get_current_time
2. time_convert_time


## Extending This to Other Free MCP Servers

You can keep the same agent and swap only the MCP tool configuration.

Examples:
- Another stdio server (open source): change `command/args` in `MCPStdioTool`
- Remote MCP endpoint: use `MCPStreamableHTTPTool` or `MCPWebsocketTool`
- OpenAPI-backed MCP server: run an MCP server that wraps an OpenAPI spec,
  then point Agent Framework to that server the same way

This keeps your agent prompts stable while changing data/capability providers.

In [63]:
await credential.close()
await mcp_time_tool.close()
print("Done!")

Done!


---

## Key Takeaways

- MCP gives agents a standard interface to external capabilities and data.
- `MCPStdioTool` is the easiest way to connect local/open-source MCP servers from notebooks.
- Tool usage can be verified by logging parsed tool results and checking observed calls.
- You can swap MCP providers (stdio, HTTP, websocket) without rewriting agent prompts.

---

**Next up:** [10 - Host MicrosoftAgent as a Non-Azure API](./10-hosting-microsoft-agent-api.ipynb) -
run an agent behind FastAPI and call it like a production API endpoint.